# Stage 11A — Dependence Sensitivity Evaluation

This notebook performs **evaluation only**. It does not retrain, fine-tune, select checkpoints, or tune thresholds.

Two prespecified sensitivity subsets are evaluated:
1. strict confirmed near-duplicate exclusion;
2. conservative direct train-scene exclusion.

All six completed YOLO11m final-epoch checkpoints and all frozen Stage 6 thresholds are retained.

In [ ]:
# Cell 1 — Mount Drive and verify GPU
from pathlib import Path
from google.colab import drive
import shutil, subprocess
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
RUN_ROOT=DRIVE/'Stage5_Baselines'
STAGE6=DRIVE/'Stage6_Heldout_Evaluation'
OUT=DRIVE/'Stage11A_Dependence_Sensitivity'
OUT.mkdir(parents=True,exist_ok=True)
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select T4 GPU before continuing.')
subprocess.run(['nvidia-smi'],check=True)
print('STAGE11A_GPU: PASS')

In [ ]:
# Cell 2 — Reassemble the same frozen h9-v2 data bundle
import hashlib, zipfile, shutil
PARTS=[DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',
       DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',
       DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02']
BUNDLE=Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED_SHA='dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h=hashlib.sha256()
with BUNDLE.open('wb') as out:
    for p in PARTS:
        if not p.is_file(): raise FileNotFoundError(p)
        with p.open('rb') as f:
            for ch in iter(lambda:f.read(8*1024*1024),b''):
                out.write(ch); h.update(ch)
assert h.hexdigest()==EXPECTED_SHA,(h.hexdigest(),EXPECTED_SHA)
DATA=Path('/content/stage11_frozen_data')
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir()
with zipfile.ZipFile(BUNDLE) as z:
    assert z.testzip() is None
    z.extractall(DATA)
Y_DATA=DATA/'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA=DATA/'h9_v2/Construction-PPE-h9-v2/data.yaml'
print('STAGE11A_FROZEN_DATA: PASS')

In [ ]:
# Cell 3 — Clone the frozen Stage 11 branch and install exact evaluation layer
import subprocess, shutil, sys
REPO=Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch','stage11-major-revision','--single-branch',
'https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.156','PyYAML==6.0.3','pandas'],check=True)
import ultralytics, torch
assert ultralytics.__version__=='8.4.156'
assert torch.cuda.is_available()
print('STAGE11A_CODE_ENV: PASS')

In [ ]:
# Cell 4 — Run both frozen sensitivity scenarios
import subprocess, sys
SCRIPT=REPO/'cross_domain_reliability/stage11_major_revision/run_stage11_sensitivity.py'
cmd=[sys.executable,str(SCRIPT),
'--repo',str(REPO),
'--stage5-root',str(RUN_ROOT),
'--y-data',str(Y_DATA),
'--c-data',str(C_DATA),
'--out',str(OUT),
'--original-group-summary',str(STAGE6/'stage6_group_summary.csv'),
'--device','0','--batch','8','--imgsz','640',
'--ap-conf','0.001','--operating-conf','0.25','--nms-iou','0.70','--match-iou','0.50']
subprocess.run(cmd,check=True)

In [ ]:
# Cell 5 — Display sensitivity comparison
import pandas as pd
from IPython.display import display
p=OUT/'stage11_sensitivity_comparison.csv'
display(pd.read_csv(p))
print('Saved to:',OUT)
print('STAGE11A_COMPLETE')